# Week 9 Lab: Natural Language Processing (Sentiment Analysis)

## Foreword
In this lab, we dive into **NLP**.
We will build a model that can read a sentence and decide if it is **Positive** or **Negative**.
We will implement the entire pipeline:
1.  **Tokenization**: Converting text to numbers.
2.  **Embeddings**: Converting numbers to vectors.
3.  **Classification**: Using PyTorch to predict sentiment.

### Step 1: Import Dependencies
We need `torch`.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

### Step 2: Define Data (Mini-IMDb)
For simplicity, we define a small dataset manually. In real life, this would be thousands of reviews.

In [ ]:
raw_data = [
    ("this movie is great", 1),
    ("i love this film", 1),
    ("fantastic acting and story", 1),
    ("terrible plot", 0),
    ("waste of time", 0),
    ("boring and slow", 0),
    ("best movie ever", 1),
    ("i hated it", 0),
    ("highly recommended", 1),
    ("do not watch", 0)
]

print(f"Dataset Size: {len(raw_data)}")
print(f"Example: {raw_data[0]}")

### Step 3: Build Vocabulary
We need to map every unique word to a unique Integer ID.

In [ ]:
word_to_ix = {"<PAD>": 0, "<UNK>": 1}

for sentence, label in raw_data:
    for word in sentence.split():
        if word not in word_to_ix:
            word_to_ix[word] = len(word_to_ix)

print(f"Vocab Size: {len(word_to_ix)}")
print("Mapping:", word_to_ix)

### Step 4: Tokenization Helper
A function to convert a sentence string into a tensor of IDs.

In [ ]:
def prepare_sequence(seq, to_ix):
    idxs = [to_ix.get(w, to_ix["<UNK>"]) for w in seq.split()]
    return torch.tensor(idxs, dtype=torch.long)

# Test it
print(prepare_sequence("this movie is great", word_to_ix))

### Step 5: Define the Model (EmbeddingBag)
We use a simple but effective architecture:
1.  **Embedding**: Converts Word IDs to Vectors (Dimensions=10).
2.  **Mean**: Averages the vectors of all words in the sentence.
3.  **Linear**: Maps the average vector to the Output (2 classes: Positive/Negative).

In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super(TextClassifier, self).__init__()
        # TODO: EmbeddingBag(vocab_size, embed_dim, sparse=False)
        self.embedding = None
        # TODO: Linear(embed_dim, num_class)
        self.fc = None

    def forward(self, text, offsets):
        embedded = self.embedding(text, offsets)
        return self.fc(embedded)

# Config
VOCAB_SIZE = len(word_to_ix)
EMBED_DIM = 10
NUM_CLASS = 2
model = TextClassifier(VOCAB_SIZE, EMBED_DIM, NUM_CLASS)
print(model)


<details>
<summary><strong>Click for Solution</strong></summary>

```python
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super(TextClassifier, self).__init__()
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, sparse=False)
        self.fc = nn.Linear(embed_dim, num_class)

    def forward(self, text, offsets):
        embedded = self.embedding(text, offsets)
        return self.fc(embedded)

# Config
VOCAB_SIZE = len(word_to_ix)
EMBED_DIM = 10
NUM_CLASS = 2
model = TextClassifier(VOCAB_SIZE, EMBED_DIM, NUM_CLASS)
print(model)```
</details>

### Step 6: Train the Model
We train for 100 epochs.

In [ ]:
optimizer = optim.SGD(model.parameters(), lr=0.1)
criterion = nn.CrossEntropyLoss()

print("Training...")
for epoch in range(100):
    total_loss = 0
    for text,label in raw_data:
        model.zero_grad()
        
        # Prepare inputs
        text_in = prepare_sequence(text, word_to_ix)
        label_in = torch.tensor([label], dtype=torch.long)
        offsets = torch.tensor([0], dtype=torch.long) # Since batch size is 1, offset is 0

        # Forward
        output = model(text_in, offsets)
        loss = criterion(output, label_in)
        
        # Backward
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch}: Loss {total_loss:.4f}")

### Step 7: Test with New Sentences
Let's try sentences the model has never seen!

In [ ]:
def predict(text):
    with torch.no_grad():
        text_in = prepare_sequence(text, word_to_ix)
        offsets = torch.tensor([0], dtype=torch.long)
        output = model(text_in, offsets)
        predicted_cls = output.argmax(1).item()
        sentiment = "POSITIVE" if predicted_cls == 1 else "NEGATIVE"
        print(f"'{text}' -> {sentiment}")

predict("this is great")
predict("absolute rubbish")
predict("i love it")